# W5D2 — RNNs, LSTMs and GRUs — Guided

**Week 5 · Day 2 · NLP Foundations** · Lab

Yesterday both models read a review as a bag of words. The order pair — `the food was good but the
service was bad` against the same words swapped — got the **same** probability from TF-IDF and almost
the same from the embeddings. Neither model can know which word came first.

Today's network reads **one token at a time, in order**, and carries a hidden state from each token to
the next. That is a recurrent neural network. You compute one by hand, measure why the plain version
forgets, see how the LSTM's gates fix that, and then train a GRU on the reviews and put it on the
leaderboard.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٥ · اليوم ٢ — الشبكات المتكرّرة وLSTM وGRU

**الأسبوع الخامس · اليوم الثاني · أساسيات معالجة اللغة** · معمل

قرأ نموذجا الأمس المراجعةَ كيسًا من الكلمات. فنال زوج الترتيب — `the food was good but the service was bad`
مقابل الكلمات نفسها مبدّلة — الاحتمال **نفسه** من TF-IDF، والاحتمال نفسه تقريبًا من التضمينات. ولا يستطيع أيّ
النموذجين أن يعرف أيّ كلمة جاءت أولًا.

وشبكة اليوم تقرأ **رمزًا رمزًا، بالترتيب**، وتحمل حالةً خفيّة (hidden state) من كل رمز إلى الذي يليه. وهذه هي
الشبكة العصبية المتكرّرة (RNN). تحسب واحدةً يدويًا، وتقيس لماذا تنسى النسخة البسيطة، وترى كيف تُصلح بوّابات
LSTM ذلك، ثم تُدرّب GRU على المراجعات وتضعها في اللوحة.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this lab you can:

- Compute an RNN step by hand and match `nn.RNN` with the same weights.
- Say what the extra weights of a GRU and an LSTM are for, from their parameter counts.
- Measure the vanishing gradient through time, and show how an open forget gate keeps it alive.
- Train recurrent models on a memory task and say how far back they can reach.
- Train a GRU text classifier with padding handled correctly, and explain the bug when it isn't.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تحسب خطوة RNN يدويًا وتطابقها مع `nn.RNN` بالأوزان نفسها.
- أن تقول فيمَ تُستعمل أوزان GRU وLSTM الإضافية، انطلاقًا من عدد معاملاتها.
- أن تقيس تلاشي التدرّج عبر الزمن، وتُظهر كيف تُبقيه بوّابة النسيان المفتوحة حيًّا.
- أن تُدرّب نماذج متكرّرة على مهمة ذاكرة وتقول إلى أيّ مدى تصل إلى الوراء.
- أن تُدرّب مُصنّف نصوص بـGRU مع معالجة الحشو صحيحًا، وتشرح الخلل حين لا تُعالجه.

</div>

## About the data

**Dataset:** `reviews_sentiment`, the same 12,000 reviews on the **same split** as yesterday:
`week5_split.json` is loaded and its hash checked, so today's row on the leaderboard is measured on
the same 2,400 test reviews as yesterday's two. If you missed D1, the reference copy in
`solutions_cache` is used.

The network reads token ids, so the reviews become integer sequences: a vocabulary of the 20,000 most
common training words, padded with `0` to 128 tokens. The longest review is under 128 tokens, so
nothing is cut. Most reviews are much shorter, and so most rows are mostly padding — which is what
task 2.4 is about.

**The memory task** in task 2.3 is synthetic: random sequences whose label is their first token. It
trains in seconds and measures one thing cleanly — how far back a model can remember.

Nothing downloads today. Every model trains on the CPU in under half a minute; task 2.3 trains nine
small ones, about a minute and a half together.

<div dir="rtl" align="right">

## عن البيانات

**مجموعة البيانات:** `reviews_sentiment`، المراجعات نفسها، ١٢٬٠٠٠ مراجعة، على **التقسيم نفسه** كالأمس: يُحمَّل
`week5_split.json` ويُتحقَّق من بصمته، فيُقاس صفّ اليوم في اللوحة على مراجعات الاختبار نفسها، ٢٬٤٠٠ مراجعة،
كصفَّي الأمس. وإن فاتك اليوم الأول فتُستعمل النسخة المرجعية في `solutions_cache`.

تقرأ الشبكة أرقام رموز، فتصير المراجعات متتاليات أعداد صحيحة: معجم من أكثر ٢٠٬٠٠٠ كلمة شيوعًا في التدريب،
محشوّة بـ`0` حتى ١٢٨ رمزًا. وأطول مراجعة أقلّ من ١٢٨ رمزًا، فلا يُقصّ شيء. ومعظم المراجعات أقصر بكثير،
فمعظم الصفوف حشوٌ في أغلبها — وهذا موضوع المهمة ٢٫٤.

**ومهمة الذاكرة** في المهمة ٢٫٣ اصطناعية: متتاليات عشوائية تسميتها رمزها الأول. تتدرّب في ثوانٍ وتقيس شيئًا
واحدًا بوضوح — إلى أيّ مدى يستطيع النموذج أن يتذكّر إلى الوراء.

لا يُنزَّل شيء اليوم. وكل نموذج يتدرّب على المعالج في أقلّ من نصف دقيقة؛ وتُدرّب المهمة ٢٫٣ تسعة نماذج صغيرة،
نحو دقيقة ونصف معًا.

</div>

## Setup

<div dir="rtl" align="right">

## الإعداد

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, versions
from aiep.data import get_dataset, load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, report

ensure("matplotlib")
seed_everything(42)

import collections
import hashlib
import json
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

SEED = 42
MAX_LEN, VOCAB_SIZE = 128, 20_000
NEGATION_PAIR = ["the service was good", "the service was not good"]
ORDER_PAIR = ["the food was good but the service was bad",
              "the service was good but the food was bad"]
torch.set_num_threads(4)
reviews = pd.read_parquet(get_dataset("reviews_sentiment"))
split = json.loads(load_artefact("week5_split.json").read_text(encoding="utf-8"))
check(hashlib.sha256(json.dumps({"train": split["train"], "test": split["test"]}).encode()).hexdigest()
      == split["sha256"], "week5_split.json must match its own hash", "يجب أن يطابق `week5_split.json` بصمته")
train, test = reviews.iloc[split["train"]], reviews.iloc[split["test"]]
print(f"train {len(train):,} | test {len(test):,} | split {split['sha256'][:12]}…")
print(versions())

In [ ]:
def load_board(days):
    """The leaderboard rows earlier days wrote (each day keeps its own file)."""
    rows = []
    for day in days:
        rows += json.loads(load_artefact(f"leaderboard_{day}.json").read_text(encoding="utf-8"))
    return rows


def show_board(rows):
    columns = ["day", "model", "n_train", "accuracy", "f1", "fit_s", "ms_per_1000", "size_mb",
               "negation_ok", "order_gap"]
    print(pd.DataFrame(rows)[columns].to_string(index=False))

## Section 1 — Warm-up: one RNN, by hand  (≈25 min)

Everything here works.

### Yesterday's problem, in numbers

The two rows yesterday wrote. Look at the `order_gap` column: 0 for TF-IDF, almost 0 for the
embeddings.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: RNN واحدة، يدويًا (نحو ٢٥ دقيقة)

كل ما هنا يعمل.

### مشكلة الأمس بالأرقام

الصفّان اللذان كتبهما الأمس. انظر عمود `order_gap`: صفر لـTF-IDF، وصفر تقريبًا للتضمينات.

</div>

In [ ]:
show_board(load_board(["D1"]))

### One step at a time

An RNN keeps one hidden vector `h`, and at each token it updates it:

`h_t = tanh(W_x · x_t + W_h · h_{t−1} + b)`

The same weights at every step — only `h` changes. Three tokens of length 2, the ones you will meet
again on Thursday: `x₁ = [1, 0]`, `x₂ = [0, 1]`, `x₃ = [1, 1]`. `W_x` is the identity, `W_h` is
`0.5 × identity`, `b = 0`, and `h₀ = [0, 0]`:

| step | inside the tanh | `h_t` |
|---|---|---|
| 1 | `[1, 0] + 0.5·[0, 0]` = `[1, 0]` | **`[0.762, 0.000]`** |
| 2 | `[0, 1] + 0.5·[0.762, 0]` = `[0.381, 1]` | **`[0.364, 0.762]`** |
| 3 | `[1, 1] + 0.5·[0.364, 0.762]` = `[1.182, 1.381]` | **`[0.827, 0.881]`** |

The code does it with a loop, then with `nn.RNN` holding the same weights. Then it feeds the tokens in
**reverse** order. The final state changes. The order is in the numbers now.

<div dir="rtl" align="right">

### خطوةً خطوة

تحتفظ RNN بمتّجه خفيّ واحد `h`، وعند كل رمز تُحدّثه:

`h_t = tanh(W_x · x_t + W_h · h_{t−1} + b)`

الأوزان نفسها في كل خطوة — وحده `h` يتغيّر. ثلاثة رموز طولها ٢، وستلقاها مرّة أخرى يوم الخميس: `x₁ = [1, 0]`
و`x₂ = [0, 1]` و`x₃ = [1, 1]`. و`W_x` مصفوفة الوحدة، و`W_h` نصف مصفوفة الوحدة، و`b = 0`، و`h₀ = [0, 0]`:

| الخطوة | داخل tanh | `h_t` |
|---|---|---|
| ١ | `[1, 0] + 0.5·[0, 0]` = `[1, 0]` | **`[0.762, 0.000]`** |
| ٢ | `[0, 1] + 0.5·[0.762, 0]` = `[0.381, 1]` | **`[0.364, 0.762]`** |
| ٣ | `[1, 1] + 0.5·[0.364, 0.762]` = `[1.182, 1.381]` | **`[0.827, 0.881]`** |

تفعل الشيفرة ذلك بحلقة، ثم بـ`nn.RNN` بالأوزان نفسها. ثم تُدخل الرموز بترتيب **معكوس**. فتتغيّر الحالة
الأخيرة. صار الترتيب في الأرقام.

</div>

In [ ]:
TOKENS = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
W_X, W_H, BIAS = torch.eye(2), 0.5 * torch.eye(2), torch.zeros(2)


def rnn_by_hand(tokens):
    h = torch.zeros(2)
    states = []
    for x in tokens:
        h = torch.tanh(W_X @ x + W_H @ h + BIAS)
        states.append(h)
    return torch.stack(states)


HAND_STATES = rnn_by_hand(TOKENS)
print("by hand:\n", HAND_STATES.numpy().round(3))

In [ ]:
rnn = nn.RNN(input_size=2, hidden_size=2, batch_first=True)
with torch.no_grad():
    rnn.weight_ih_l0.copy_(W_X)
    rnn.weight_hh_l0.copy_(W_H)
    rnn.bias_ih_l0.zero_()
    rnn.bias_hh_l0.zero_()

outputs, last = rnn(TOKENS.unsqueeze(0))          # (batch=1, seq=3, features=2)
print("nn.RNN outputs:", tuple(outputs.shape), "\n", outputs[0].detach().numpy().round(3))
REVERSED_LAST = rnn(TOKENS.flip(0).unsqueeze(0))[1][0, 0]
print("\nlast state, in order :", last[0, 0].detach().numpy().round(3))
print("last state, reversed :", REVERSED_LAST.detach().numpy().round(3))

## Section 2 — Core: five tasks  (≈60 min)

1. Count the parameters of an RNN, a GRU and an LSTM.
2. Measure the gradient through time — and keep it alive with the forget gate.
3. The memory task: how far back can they remember?
4. A GRU on the reviews, and the padding trap.
5. The leaderboard row, the two sentence pairs, and long reviews.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: خمس مهام (نحو ٦٠ دقيقة)

١. عُدّ معاملات RNN وGRU وLSTM.
٢. قِس التدرّج عبر الزمن — وأبقِه حيًّا ببوّابة النسيان.
٣. مهمة الذاكرة: إلى أيّ مدى تتذكّر؟
٤. GRU على المراجعات، وفخّ الحشو.
٥. صفّ اللوحة، وزوجا الجمل، والمراجعات الطويلة.

</div>

### Task 2.1 — three cells, three sizes

Build `nn.RNN`, `nn.GRU` and `nn.LSTM`, each with 64 inputs and 64 hidden units, and count their
parameters. The ratio is exactly **1 : 3 : 4**. A plain RNN has one block of weights — the update. A
GRU has three: the update and two gates (*reset*, *update*). An LSTM has four: the update and three
gates (*input*, *forget*, *output*), plus a second state vector, the **cell**, that the gates protect.
More parameters per step is the price of a memory that can last.

<div dir="rtl" align="right">

### المهمة ٢٫١ — ثلاث خلايا، وثلاثة أحجام

ابنِ `nn.RNN` و`nn.GRU` و`nn.LSTM`، لكلٍّ ٦٤ مدخلًا و٦٤ وحدة خفيّة، وعُدّ معاملاتها. النسبة **١ : ٣ : ٤**
بالضبط. لـRNN البسيطة كتلة أوزان واحدة — التحديث. ولـGRU ثلاث: التحديث وبوّابتان (*إعادة الضبط* و*التحديث*).
ولـLSTM أربع: التحديث وثلاث بوّابات (*الإدخال* و*النسيان* و*الإخراج*)، ومعها متّجه حالة ثانٍ، هو **الخلية**،
تحميه البوّابات. وزيادة المعاملات في كل خطوة ثمنُ ذاكرةٍ قادرة على البقاء.

</div>

In [ ]:
# TODO: Build the three cells (64 in, 64 hidden) and count their parameters.
# مهمة: ابنِ الخلايا الثلاث (٦٤ مدخلًا و٦٤ وحدة خفيّة) وعُدّ معاملاتها.
for name, n in PARAMS.items():
    print(f"{name:<5} {n:>6,} parameters  ({n / PARAMS['RNN']:.0f}x the RNN)")

### Task 2.2 — the gradient through time

To learn from the first token, the gradient has to travel back through every step. Each step
multiplies it by a matrix, so over T steps it shrinks — or grows — geometrically. Measure it: feed a
random sequence of length T, take the gradient of the **last** output with respect to the **first**
input, and record its size for T = 10, 25, 50 and 100.

Three networks, the same 32 hidden units:

- a plain `nn.RNN` — the gradient is gone after a few dozen steps;
- an `nn.LSTM` as PyTorch initialises it — it is gone too;
- the same LSTM with its **forget gate opened**: the gate's bias set to +4, so the gate starts near 1
  and the cell copies itself forward.

The third one barely decays. That is the LSTM's whole idea: the cell state is a path along which the
gradient is multiplied by the forget gate and nothing else, and the network can *learn* to keep it open.
Opening it at the start, as here, is a standard trick.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — التدرّج عبر الزمن

ليتعلّم النموذج من الرمز الأول، يجب أن يسافر التدرّج إلى الوراء عبر كل خطوة. وكل خطوة تضربه في مصفوفة،
فيصغر — أو يكبر — هندسيًا على مدى T خطوة. فقِسه: أدخِل متتالية عشوائية طولها T، وخُذ تدرّج المخرَج **الأخير**
بالنسبة إلى المُدخَل **الأول**، وسجّل حجمه عند T = ١٠ و٢٥ و٥٠ و١٠٠.

ثلاث شبكات، بالوحدات الخفيّة نفسها، ٣٢ وحدة:

- `nn.RNN` بسيطة — يختفي التدرّج بعد بضع عشرات من الخطوات؛
- `nn.LSTM` كما يهيّئها PyTorch — ويختفي أيضًا؛
- LSTM نفسها و**بوّابة النسيان مفتوحة**: انحياز البوّابة +٤، فتبدأ البوّابة قرب ١ وتنسخ الخلية نفسها إلى الأمام.

والثالثة بالكاد تتلاشى. وهذه فكرة LSTM كلها: حالة الخلية طريقٌ يُضرب فيه التدرّج في بوّابة النسيان ولا شيء
غيرها، والشبكة تستطيع أن *تتعلّم* إبقاءها مفتوحة. وفتحها في البداية، كما هنا، حيلة شائعة.

</div>

In [ ]:
def open_forget_gate(lstm, bias=4.0):
    """PyTorch packs the LSTM biases as [input, forget, cell, output]; set the forget slice."""
    for name, p in lstm.named_parameters():
        if "bias" in name:
            hidden = p.shape[0] // 4
            with torch.no_grad():
                p[hidden:2 * hidden] = bias / 2        # bias_ih and bias_hh are added together


def first_input_gradient(network, length, seeds=range(5)):
    """Mean size of d(last output) / d(first input), over a few random sequences."""
    # TODO: For each seed, run a random (1, length, 8) input with requires_grad, backprop the sum of the last output, and keep the norm of the gradient at position 0. Return the mean.
    # مهمة: لكل بذرة، شغّل مُدخلًا عشوائيًا `(1, length, 8)` مع `requires_grad`، وانشر مجموع المخرَج الأخير إلى الوراء، واحتفظ بمعيار التدرّج عند الموضع ٠. وأعِد المتوسط.

In [ ]:
LENGTHS = [10, 25, 50, 100]
torch.manual_seed(SEED)
networks = {"RNN": nn.RNN(8, 32, batch_first=True), "LSTM": nn.LSTM(8, 32, batch_first=True),
            "LSTM, forget gate open": nn.LSTM(8, 32, batch_first=True)}
open_forget_gate(networks["LSTM, forget gate open"])

GRADIENTS = pd.DataFrame({name: [first_input_gradient(net, T) for T in LENGTHS]
                          for name, net in networks.items()}, index=LENGTHS)
print(GRADIENTS.map(lambda v: f"{v:.1e}").to_string())
GRADIENTS.plot(logy=True, marker="o", title="size of the gradient reaching the first token")
plt.xlabel("sequence length")
plt.show()

### Task 2.3 — the memory task

Random sequences of tokens 4–11, except the first, which is a label 0–3. The model reads the whole
sequence and must answer with the first token. Chance is 0.25.

Train a plain RNN, a GRU and an LSTM (forget gate open) for the same 800 steps: once on sequences of
length **25**, and twice — with two different seeds — on length **100**.

All three remember across 25 tokens. At 100 the table becomes a lottery: some runs learn the task
perfectly and others sit at chance, and **which ones** changes with the seed. That is the practical
face of the vanishing gradient. The gates make success *possible* (task 2.2), but the signal from the
first token has to be found through 100 steps of noise, and whether the optimiser finds it in time
depends on where it started. Keep the table: Wednesday's and Thursday's architectures take the same
test.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — مهمة الذاكرة

متتاليات عشوائية من الرموز ٤ إلى ١١، إلا الأول فهو تسمية من ٠ إلى ٣. يقرأ النموذج المتتالية كلها وعليه أن يُجيب
بالرمز الأول. والصدفة ٠٫٢٥.

درّب RNN بسيطة وGRU وLSTM (بوّابة النسيان مفتوحة) ٨٠٠ خطوة مرّةً على متتاليات طولها ٢٥، ومرّتين — ببذرتين مختلفتين — على الطول ١٠٠.

تتذكّر الثلاث عبر ٢٥ رمزًا. وعند ١٠٠ يصير الجدول يانصيبًا: بعض التشغيلات يتعلّم المهمة تمامًا وبعضها يثبت عند
الصدفة، و**أيّها** يتغيّر بتغيّر البذرة. وهذا هو الوجه العملي لتلاشي التدرّج. فالبوّابات تجعل النجاح *ممكنًا*
(المهمة ٢٫٢)، لكن إشارة الرمز الأول يجب أن تُوجَد عبر ١٠٠ خطوة من الضجيج، وهل يجدها المُحسِّن في الوقت أم لا
يتوقّف على نقطة بدايته. احتفظ بالجدول: فمعماريّتا الأربعاء والخميس تخوضان الاختبار نفسه.

</div>

In [ ]:
# The memory task: the label is the first token; every token after it is random noise.
N_CLASSES, MEMORY_VOCAB = 4, 12          # tokens 0-3 can be labels, 4-11 are distractors


def memory_batch(n, length, generator):
    x = torch.randint(N_CLASSES, MEMORY_VOCAB, (n, length), generator=generator)
    y = torch.randint(0, N_CLASSES, (n,), generator=generator)
    x[:, 0] = y
    return x, y


def train_memory(model, length, steps=800, lr=0.01, seed=0):
    """Train on fresh random batches, then score 1,000 held-out sequences (chance is 0.25)."""
    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        x, y = memory_batch(64, length, generator)
        loss = nn.functional.cross_entropy(model(x), y)
        optimiser.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
    x, y = memory_batch(1000, length, torch.Generator().manual_seed(999))
    with torch.no_grad():
        return float((model(x).argmax(1) == y).float().mean())

In [ ]:
class RecurrentMemory(nn.Module):
    """Embed the tokens, run a recurrent cell, read the answer off the last hidden state."""

    def __init__(self, cell):
        super().__init__()
        self.embed = nn.Embedding(MEMORY_VOCAB, 16)
        self.cell = getattr(nn, cell)(16, 64, batch_first=True)
        if cell == "LSTM":
            open_forget_gate(self.cell)
        self.out = nn.Linear(64, N_CLASSES)

    def forward(self, x):
        # TODO: Embed, run the cell, and classify from the output at the last time step.
        # مهمة: ضمِّن، وشغّل الخلية، وصنّف من المخرَج عند آخر خطوة زمنية.

In [ ]:
MEMORY_RESULTS = {}
for cell in ["RNN", "GRU", "LSTM"]:
    for length, seed in [(25, 0), (100, 0), (100, 1)]:
        start = time.perf_counter()
        torch.manual_seed(seed)
        key = f"{cell} @ {length} (seed {seed})"
        MEMORY_RESULTS[key] = train_memory(RecurrentMemory(cell), length, seed=seed)
        print(f"{key:<22} accuracy {MEMORY_RESULTS[key]:.3f}  ({time.perf_counter() - start:.0f} s)")

### Task 2.4 — a GRU on the reviews, and the padding trap

The reviews become token ids, right-padded with 0 to 128 (the cell below does that). The model is
`Embedding → GRU → Linear`.

First the **trap**. Read the answer off `output[:, -1]` — the last time step — as in task 2.3. That
worked there because every sequence had the same length. Here the last time step of a 20-word review
is **108 steps of padding later**, and the GRU's state has been overwritten by them. Train it for one
epoch and watch the accuracy sit at 0.50.

Then the fix: `pack_padded_sequence` with the true lengths, so the GRU stops at each review's real
last word, and read its final hidden state. Three epochs, about 20 seconds.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — GRU على المراجعات، وفخّ الحشو

تصير المراجعات أرقام رموز، محشوّة من اليمين بـ`0` حتى ١٢٨ (والخلية أدناه تفعل ذلك). والنموذج
`Embedding ← GRU ← Linear`.

أولًا **الفخّ**. اقرأ الجواب من `output[:, -1]` — آخر خطوة زمنية — كما في المهمة ٢٫٣. نجح ذلك هناك لأن
المتتاليات كلها بالطول نفسه. أمّا هنا فآخر خطوة زمنية لمراجعة من عشرين كلمة تأتي **بعد ١٠٨ خطوات من الحشو**،
وقد محت تلك الخطوات حالة GRU. درّبه حقبة واحدة وراقب الدقّة تثبت عند ٠٫٥٠.

ثم الإصلاح: `pack_padded_sequence` بالأطوال الحقيقية، فيتوقّف GRU عند آخر كلمة حقيقية في كل مراجعة، واقرأ
حالته الخفيّة الأخيرة. ثلاث حقب، نحو عشرين ثانية.

</div>

In [ ]:
def tokenize(text):
    return re.sub(r"[^0-9a-z؀-ۿ\s']+", " ", text.lower()).split()


counts = collections.Counter(w for text in train.review_text for w in tokenize(text))
ITOS = ["<pad>", "<unk>"] + [w for w, _ in counts.most_common(VOCAB_SIZE - 2)]
STOI = {w: i for i, w in enumerate(ITOS)}


def encode(texts):
    """Token ids, right-padded with 0 to MAX_LEN, and the true length of each review."""
    ids = [[STOI.get(w, 1) for w in tokenize(t)][:MAX_LEN] or [1] for t in texts]
    x = torch.zeros(len(ids), MAX_LEN, dtype=torch.long)
    for i, seq in enumerate(ids):
        x[i, :len(seq)] = torch.tensor(seq)
    return x, torch.tensor([len(seq) for seq in ids])


X_train, len_train = encode(train.review_text)
X_test, len_test = encode(test.review_text)
y_train = torch.tensor(train.sentiment.to_numpy()).long()
y_test = torch.tensor(test.sentiment.to_numpy()).long()
print(f"vocabulary {len(ITOS):,} | X_train {tuple(X_train.shape)} | longest review {int(len_train.max())} tokens")

In [ ]:
class ReviewGRU(nn.Module):
    def __init__(self, pack=True, cell="GRU", bidirectional=False):
        super().__init__()
        self.pack = pack
        self.embed = nn.Embedding(len(ITOS), 64, padding_idx=0)
        self.cell = getattr(nn, cell)(64, 64, batch_first=True, bidirectional=bidirectional)
        self.out = nn.Linear(64 * (2 if bidirectional else 1), 2)

    def forward(self, x, lengths):
        if not self.pack:                                   # the trap: the last step is padding
            states, _ = self.cell(self.embed(x))
            return self.out(states[:, -1])
        # TODO: Pack the embedded batch with the true lengths, run the cell, and classify from the final hidden state (for an LSTM it is the first of a pair; for two directions, concatenate both).
        # مهمة: احزم الدفعة المُضمَّنة بالأطوال الحقيقية، وشغّل الخلية، وصنّف من الحالة الخفيّة الأخيرة (وفي LSTM هي الأولى من زوج؛ وللاتجاهين اجمعهما متجاورتين).


def fit_reviews(model, epochs, lr=2e-3):
    """Mini-batches of 64 over the training set; returns the seconds it took."""
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    start = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(X_train))
        for i in range(0, len(order), 64):
            batch = order[i:i + 64]
            loss = nn.functional.cross_entropy(model(X_train[batch], len_train[batch]), y_train[batch])
            optimiser.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimiser.step()
        print(f"  epoch {epoch + 1}: test accuracy {accuracy(model):.4f}")
    return time.perf_counter() - start


def accuracy(model):
    model.eval()
    with torch.no_grad():
        return float((model(X_test, len_test).argmax(1) == y_test).float().mean())

In [ ]:
torch.manual_seed(SEED)
trap_model = ReviewGRU(pack=False)
print("last time step, no packing:")
fit_reviews(trap_model, epochs=1)
TRAP_ACCURACY = accuracy(trap_model)

In [ ]:
torch.manual_seed(SEED)
gru = ReviewGRU(pack=True)
print("packed, final hidden state:")
GRU_FIT_S = fit_reviews(gru, epochs=3)
GRU_ACCURACY = accuracy(gru)
print(f"\n{GRU_ACCURACY:.4f} against {TRAP_ACCURACY:.4f} for the same model reading padding — in {GRU_FIT_S:.0f} s")

### Task 2.5 — the leaderboard row, both pairs, and long reviews

Score the GRU like yesterday's models: accuracy, F1, fit time, time per 1,000 test reviews, size, the
negation pair and the order pair. The `order_gap` is not zero any more. It is small, but this is the
first model this week that can tell the two sentences apart at all.

Then split the test reviews into three length buckets and score each. Long reviews come out a few
points worse: the state is a fixed-size summary, and a long review has to fit into it as well as a short
one. That is the bottleneck the lecture named, and the reason for the next two days.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — صفّ اللوحة، والزوجان، والمراجعات الطويلة

قيّم GRU كما قيّمت نموذجَي الأمس: الدقّة، وF1، وزمن التدريب، والزمن لكل ألف مراجعة اختبار، والحجم، وزوج النفي
وزوج الترتيب. لم يعُد `order_gap` صفرًا. إنه صغير، لكن هذا أول نموذج هذا الأسبوع يستطيع أن يميّز الجملتين
أصلًا.

ثم قسّم مراجعات الاختبار إلى ثلاث فئات بحسب الطول وقيّم كلًّا منها. تخرج المراجعات الطويلة أسوأ ببضع نقاط:
فالحالة ملخّص ثابت الحجم، وعلى المراجعة الطويلة أن تتّسع فيه كما تتّسع القصيرة. وهذا هو عنق الزجاجة الذي سمّته
المحاضرة، وهو سبب اليومين التاليين.

</div>

In [ ]:
from sklearn.metrics import f1_score


def p_positive(model, sentences):
    model.eval()
    x, lengths = encode(sentences)
    with torch.no_grad():
        return torch.softmax(model(x, lengths), dim=1)[:, 1].numpy()


# TODO: Time the predictions on the test set, and get p(positive) for both sentence pairs.
# مهمة: وقّت التنبّؤ على مجموعة الاختبار، واحسب `p(positive)` لزوجَي الجمل.

BOARD_D2 = [{
    "day": "D2", "model": "GRU (trained from scratch)", "n_train": len(X_train),
    "accuracy": round(GRU_ACCURACY, 4), "f1": round(f1_score(y_test.numpy(), gru_predictions), 4),
    "fit_s": round(GRU_FIT_S, 1), "ms_per_1000": round(GRU_MS_PER_1000, 1),
    "size_mb": round(sum(p.numel() for p in gru.parameters()) * 4 / 1e6, 2),
    "negation_ok": bool(p_pairs[0] > 0.5 > p_pairs[1]), "order_gap": round(float(abs(p_pairs[2] - p_pairs[3])), 4),
}]
for sentence, p in zip(NEGATION_PAIR + ORDER_PAIR, p_pairs):
    print(f"  p(positive) = {p:.4f}   {sentence}")
print()
show_board(load_board(["D1"]) + BOARD_D2)

In [ ]:
buckets = pd.cut(len_test.numpy(), bins=[0, 40, 80, MAX_LEN], labels=["short (<40)", "medium", "long (80+)"])
BY_LENGTH = pd.Series(gru_predictions == y_test.numpy()).groupby(buckets, observed=True).agg(["size", "mean"])
print(BY_LENGTH.rename(columns={"size": "reviews", "mean": "accuracy"}).round(3).to_string())

## Section 3 — Stretch  (≈30 min)

The same classifier with three other cells: an **LSTM**, a **bidirectional GRU** (one GRU reads
left to right, another right to left, and the two final states are concatenated) and a
**bidirectional LSTM**. The `ReviewGRU` class already takes `cell=` and `bidirectional=`. Train each
for three epochs and compare accuracy, fit time and size. A bidirectional model costs twice as much and
sees each word with both its past and its future. Is it worth it here?

<div dir="rtl" align="right">

## القسم الثالث — إضافي (نحو ٣٠ دقيقة)

المُصنّف نفسه بثلاث خلايا أخرى: **LSTM**، و**GRU ثنائية الاتجاه** (تقرأ GRU من اليسار إلى اليمين، وأخرى من
اليمين إلى اليسار، وتُوصَل الحالتان الأخيرتان)، و**LSTM ثنائية الاتجاه**. والصنف `ReviewGRU` يأخذ أصلًا
`cell=` و`bidirectional=`. درّب كلًّا ثلاث حقب وقارن الدقّة وزمن التدريب والحجم. النموذج ثنائي الاتجاه يكلّف
ضعفين ويرى كل كلمة مع ماضيها ومستقبلها. فهل يستحق ذلك هنا؟

</div>

In [ ]:
STRETCH = {}
for name, kwargs in {"LSTM": dict(cell="LSTM"), "bidirectional GRU": dict(bidirectional=True),
                     "bidirectional LSTM": dict(cell="LSTM", bidirectional=True)}.items():
    torch.manual_seed(SEED)
    model = ReviewGRU(**kwargs)
    print(name)
    seconds = fit_reviews(model, epochs=3)
    STRETCH[name] = {"accuracy": round(accuracy(model), 4), "fit_s": round(seconds, 1),
                     "params": sum(p.numel() for p in model.parameters())}
print(pd.DataFrame(STRETCH).T.to_string())

## Save your artefacts

- `leaderboard_D2.json` — today's row.
- `memory_task_D2.json` — the memory-task table. D3 and D4 add their architectures to it.

<div dir="rtl" align="right">

## احفظ آثارك

- `leaderboard_D2.json` — صفّ اليوم.
- `memory_task_D2.json` — جدول مهمة الذاكرة. ويضيف إليه اليومان الثالث والرابع معماريّتيهما.

</div>

In [ ]:
(ARTEFACT_DIR / "leaderboard_D2.json").write_text(json.dumps(BOARD_D2, indent=2), encoding="utf-8")
(ARTEFACT_DIR / "memory_task_D2.json").write_text(json.dumps(MEMORY_RESULTS, indent=2), encoding="utf-8")
print(json.dumps(MEMORY_RESULTS, indent=2))

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check(torch.allclose(HAND_STATES, outputs[0], atol=1e-6)
      and torch.allclose(HAND_STATES[-1], torch.tensor([0.827, 0.881]), atol=1e-3),
      "the hand-computed states must match nn.RNN, ending at [0.827, 0.881]",
      "يجب أن تُطابق الحالات المحسوبة يدويًا `nn.RNN`، وتنتهي عند `[0.827, 0.881]`")
check(not torch.allclose(REVERSED_LAST, last[0, 0]),
      "reversing the tokens must change the final state — the RNN sees order",
      "يجب أن يُغيّر عكسُ الرموز الحالة الأخيرة — فـRNN ترى الترتيب")
check(PARAMS["GRU"] == 3 * PARAMS["RNN"] and PARAMS["LSTM"] == 4 * PARAMS["RNN"],
      f"the parameter ratio must be exactly 1 : 3 : 4 — got {PARAMS}",
      f"يجب أن تكون نسبة المعاملات ١ : ٣ : ٤ بالضبط — والناتج {PARAMS}")
check(GRADIENTS.loc[100, "RNN"] < 1e-8 and GRADIENTS.loc[100, "LSTM, forget gate open"] > 1e-2,
      "at length 100 the RNN's gradient must have vanished and the open-gate LSTM's must not",
      "عند الطول ١٠٠ يجب أن يكون تدرّج RNN قد تلاشى وألّا يتلاشى تدرّج LSTM ذات البوّابة المفتوحة")
at_100 = [v for k, v in MEMORY_RESULTS.items() if "@ 100" in k]
check(all(MEMORY_RESULTS[f"{c} @ 25 (seed 0)"] > 0.9 for c in ["RNN", "GRU", "LSTM"]) and min(at_100) < 0.5,
      f"every cell must solve the memory task at length 25, and at least one length-100 run must fail — got {MEMORY_RESULTS}",
      f"يجب أن تحلّ كل خلية مهمة الذاكرة عند الطول ٢٥، وأن يفشل تشغيل واحد على الأقل عند ١٠٠ — والناتج {MEMORY_RESULTS}")
check(TRAP_ACCURACY < 0.6 and GRU_ACCURACY > 0.75,
      f"reading padding must leave the GRU near chance ({TRAP_ACCURACY:.3f}) and packing must fix it ({GRU_ACCURACY:.3f})",
      f"يجب أن تُبقي قراءةُ الحشو GRU قرب الصدفة ({TRAP_ACCURACY:.3f}) وأن يُصلحه الحزم ({GRU_ACCURACY:.3f})")
check(BOARD_D2[0]["order_gap"] > 0,
      "the GRU must give the two order-pair sentences different probabilities",
      "يجب أن تُعطي GRU جملتَي زوج الترتيب احتمالين مختلفين")
report()

## What's next

**W5D3 — Temporal convolutional networks.** The RNN reads one token after another, so it cannot be
parallelised along the sequence, and on the memory task every cell fell to chance at length 100.

Tomorrow's network reads a sequence with **convolutions** instead: a filter slides along the tokens,
and stacks of dilated filters see further back with every layer. No step-by-step loop, so it runs in
parallel. Its memory has a hard edge — the receptive field — and you will compute it, measure it, and
put it through the same memory task.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع ٥ اليوم ٣ — الشبكات الالتفافية الزمنية (TCN).** تقرأ RNN رمزًا بعد رمز، فلا يمكن توزيعها على
التوازي على طول المتتالية، وفي مهمة الذاكرة هبطت كل خلية إلى الصدفة عند الطول ١٠٠.

وشبكة الغد تقرأ المتتالية **بالالتفاف** بدلًا من ذلك: مُرشِّح ينزلق على الرموز، وأكوام من المرشّحات المتّسعة
(dilated) ترى أبعد إلى الوراء مع كل طبقة. لا حلقة خطوة بخطوة، فتعمل على التوازي. ولذاكرتها حدٌّ صارم — مجال
الاستقبال (receptive field) — وستحسبه وتقيسه وتُدخلها مهمة الذاكرة نفسها.

</div>